# LFM2.5-Audio × vLLM-Omni — itération GPU (Colab)

Poste de travail pour valider le plugin out-of-tree `vllm_omni_lfm2_audio`
(repo [`rcarvalo/finetuning_s2s_toolcalling`](https://github.com/rcarvalo/finetuning_s2s_toolcalling), branche `claude/blissful-tesla-7i1yky`).

Boucle d'itération : éditer en local → push → **cellule 2** (pull) → **cellule 4** (smoke) → lire le diagnostic → recommencer.

Étapes du smoke (`scripts/colab_smoke_vllm_omni.py`) : `imports` → `plugin` → `contract` → `checkpoint` → `engine`.
Critère bloquant P2 : parité greedy avec `liquid_audio.generate_interleaved` (`tests/test_omni_parity.py`).

In [ ]:
# 1. GPU
!nvidia-smi

In [ ]:
# 2. Repo (clone idempotent + pull)
import os
BRANCH = "claude/blissful-tesla-7i1yky"
REPO = "https://github.com/rcarvalo/finetuning_s2s_toolcalling.git"
if not os.path.isdir("/content/finetuning_s2s_toolcalling"):
    !git clone -b {BRANCH} {REPO} /content/finetuning_s2s_toolcalling
%cd /content/finetuning_s2s_toolcalling
!git pull --ff-only
!git log --oneline -3

In [ ]:
# 3. Dépendances (sys.executable -m pip : le kernel Colab, pas un autre python)
import sys
!{sys.executable} -m pip install -q "vllm-omni==0.22.0" "liquid-audio>=1.3.0"
!{sys.executable} -m pip install -q -e . --no-deps
import importlib.metadata as md
print("vllm-omni", md.version("vllm-omni"), "| liquid-audio", md.version("liquid-audio"))

In [ ]:
# 4. Smoke progressif (sans checkpoint : imports + plugin + contrat runtime)
import sys
!{sys.executable} scripts/colab_smoke_vllm_omni.py

In [ ]:
# 5. Checkpoint : base LFM2.5-Audio convertie au layout vLLM-Omni
#    (suffisant pour valider chargement/engine ; le finetuné FR viendra après)
import sys
from huggingface_hub import snapshot_download
base = snapshot_download("LiquidAI/LFM2.5-Audio-1.5B")
!{sys.executable} -m vllm_omni_lfm2_audio.convert_checkpoint --checkpoint {base} --output /content/lfm25_audio_omni
!{sys.executable} scripts/colab_smoke_vllm_omni.py --checkpoint /content/lfm25_audio_omni

In [ ]:
# 6. Engine (le gros test : Omni(...) démarre et génère)
import sys
!{sys.executable} scripts/colab_smoke_vllm_omni.py --checkpoint /content/lfm25_audio_omni --engine

In [ ]:
# 7. Parité P2 (bloquant) — une fois l'engine fonctionnel
import sys
!OMNI_CHECKPOINT=/content/lfm25_audio_omni BASE_MODEL=LiquidAI/LFM2.5-Audio-1.5B \
  {sys.executable} -m pytest tests/test_omni_parity.py -m gpu -q -x